# XGBoost Model

This notebook trains the XGBoost model for predicting clinical trials. It uses the same cleaned feature set as Random Forest so the reported performance is based on features plausibly available near trial start.

Input: `data/model_ready.parquet`  
Output: `artifacts/models/xgboost.joblib` and `artifacts/model_metrics.json`

## Justification for model

We chose XGBoost because gradient-boosted trees tend to perform strongly on structured tabular data and can give us a higher-capacity model to compare against the baseline and random forest.

## 1. Setup

In [1]:
import time

import pandas as pd
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

from modeling_utils import (
    LEAKAGE_PRONE_COLS,
    RANDOM_STATE,
    evaluate_model,
    load_model_ready,
    remove_leakage_prone_features,
    save_model,
    save_or_update_metrics,
    temporal_split,
)

## 2. Load Clean Model-Ready Data

In [2]:
df, feature_cols = load_model_ready()
clean_feature_cols = remove_leakage_prone_features(feature_cols)
x_train, x_test, y_train, y_test = temporal_split(df, clean_feature_cols)

sample_weight = compute_sample_weight("balanced", y_train)

print(f"Loaded: {df.shape[0]:,} rows")
print(f"Original feature count: {len(feature_cols):,}")
print(f"Clean feature count:    {len(clean_feature_cols):,}")
print(f"Removed leakage-prone columns: {sorted(LEAKAGE_PRONE_COLS)}")
print(f"Train: {x_train.shape[0]:,} rows")
print(f"Test:  {x_test.shape[0]:,} rows")

Loaded: 53,628 rows
Original feature count: 147
Clean feature count:    144
Removed leakage-prone columns: ['enrollment_actual', 'log_enrollment', 'trial_duration_days']
Train: 40,209 rows
Test:  13,419 rows


## 3. Train XGBoost

The main tuning knobs are `n_estimators`, `max_depth`, `learning_rate`, `subsample`, `colsample_bytree`, and `reg_lambda`. We evaluate with macro F1 so the minority `WITHDRAWN` class matters.

In [3]:
xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=400,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.5,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=1,
)

start = time.perf_counter()
xgb_model.fit(x_train, y_train, sample_weight=sample_weight)
training_seconds = time.perf_counter() - start

print(f"Training complete in {training_seconds:.1f} seconds")

Training complete in 20.0 seconds


## 4. Evaluate and Save

In [4]:
metrics = evaluate_model(
    model=xgb_model,
    name="XGBoost",
    slug="xgboost",
    x_train=x_train,
    x_test=x_test,
    y_test=y_test,
    training_seconds=training_seconds,
)

save_model(xgb_model, "xgboost")
save_or_update_metrics(metrics)

print(f"Macro F1:    {metrics['macro_f1']:.4f}")
print(f"Accuracy:    {metrics['accuracy']:.4f}")
print(f"Weighted F1: {metrics['weighted_f1']:.4f}")
for label in ["COMPLETED", "TERMINATED", "WITHDRAWN"]:
    scores = metrics["classification_report"][label]
    print(f"{label}: precision={scores['precision']:.4f}, recall={scores['recall']:.4f}, f1={scores['f1-score']:.4f}")
print("Saved XGBoost artifacts.")

Macro F1:    0.5653
Accuracy:    0.5810
Weighted F1: 0.5819
COMPLETED: precision=0.7496, recall=0.4683, f1=0.5764
TERMINATED: precision=0.5714, recall=0.7163, f1=0.6357
WITHDRAWN: precision=0.4089, recall=0.5926, f1=0.4839
Saved XGBoost artifacts.


## 5. Optional Hyperparameter Tuning

This cell runs grid search, random search, and Bayesian tuning for XGBoost using the standalone tuning script. The script writes `artifacts/xgboost_tuning_results.csv`, `artifacts/xgboost_tuning_summary.json`, and `artifacts/models/xgboost_tuned.joblib`.

In [ ]:
import subprocess
import sys

tuning_command = [
    sys.executable,
    "xgboost_hyperparameter_tuning.py",
    "--methods",
    "grid",
    "random",
    "bayesian",
    "--random-iterations",
    "12",
    "--bayesian-trials",
    "20",
]

subprocess.run(tuning_command, check=True)

## 6. XGBoost Feature Importance

In [ ]:
xgb_importances = pd.DataFrame({
    "feature": clean_feature_cols,
    "importance": xgb_model.feature_importances_,
}).sort_values("importance", ascending=False)

xgb_importances.head(20)

,feature,importance
30,num_sites,0.078918
43,healthy_volunteers,0.070211
35,sponsor_class_OTHER,0.055155
25,masking_UNKNOWN,0.038050
32,sponsor_class_INDUSTRY,0.023861
22,masking_NONE,0.018385
45,is_fda_regulated_drug,0.016349
44,has_dmc,0.016056
34,sponsor_class_NIH,0.014640
4,phases_PHASE2,0.014462
